```markdown
# Proyecto de Clasificación de Iris con Redes Neuronales y Dashboard Interactivo en Gradio

Este notebook contiene:
1. **EDA Completo** del dataset de Iris.
2. **Entrenamiento de una Red Neuronal** multicapa para clasificar especies.
3. **Evaluación** del modelo (Matriz de Confusión, Reporte e Histograma de Errores).
4. **Dashboard Interactivo en Gradio** que consolida todo el análisis, gráficas, modelo en tiempo real y conclusiones.
```

In [ ]:
# Instalación de dependencias necesarias
!pip install -q gradio pandas numpy scikit-learn tensorflow matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import gradio as gr

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.utils import to_categorical

# Desactivar advertencias para limpieza visual
import warnings
warnings.filterwarnings('ignore')

```markdown
## 1. Carga de Datos y EDA Preparatorio
Cargamos el dataset oficial de Iris y construimos el DataFrame.
```

In [ ]:
# Cargar datos de Iris
iris_data = load_iris()
df = pd.DataFrame(data=iris_data.data, columns=iris_data.feature_names)
df['species'] = iris_data.target

# Mapear códigos numéricos a nombres de especies reales
species_map = {i: name for i, name in enumerate(iris_data.target_names)}
df['species_name'] = df['species'].map(species_map)

# Mostrar estructura y primeras líneas
display(df.head())
display(df.describe())

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),species,species_name
0,5.1,3.5,1.4,0.2,0,setosa
1,4.9,3.0,1.4,0.2,0,setosa
2,4.7,3.2,1.3,0.2,0,setosa
3,4.6,3.1,1.5,0.2,0,setosa
4,5.0,3.6,1.4,0.2,0,setosa


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),species
count,150.000000,150.000000,150.000000,150.000000,150.000000
mean,5.843333,3.057333,3.758000,1.199333,1.000000
std,0.828066,0.435866,1.765298,0.762238,0.819232
min,4.300000,2.000000,1.000000,0.100000,0.000000
25%,5.100000,2.800000,1.600000,0.300000,0.000000
50%,5.800000,3.000000,4.350000,1.300000,1.000000
75%,6.400000,3.300000,5.100000,1.800000,2.000000
max,7.900000,4.400000,6.900000,2.500000,2.000000


```markdown
## 2. Entrenamiento de la Red Neuronal
Preparamos los datos, entrenamos un clasificador secuencial y generamos las métricas de evaluación solicitadas.
```

In [ ]:
# Preparación de datos
X = df[iris_data.feature_names].values
y = df['species'].values

# Dividir en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Escalar características
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# One-hot encoding para las etiquetas
y_train_categorical = to_categorical(y_train)
y_test_categorical = to_categorical(y_test)

# Construcción del modelo de Red Neuronal
model = Sequential([
    Dense(16, activation='relu', input_shape=(4,)),
    Dense(12, activation='relu'),
    Dense(3, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Entrenar el modelo
history = model.fit(X_train_scaled, y_train_categorical, validation_split=0.1, epochs=80, batch_size=8, verbose=0)

```markdown
## 3. Generación de Métricas y Evaluación del Modelo
Visualizaremos las métricas obtenidas incluyendo la matriz de confusión, el reporte de clasificación y el histograma de errores.
```

In [ ]:
# Predicciones
y_pred_prob = model.predict(X_test_scaled)
y_pred = np.argmax(y_pred_prob, axis=1)

# Reporte de clasificación
report = classification_report(y_test, y_pred, target_names=iris_data.target_names)
print("Reporte de Clasificación:")
print(report)

# Histograma de errores (diferencia entre la probabilidad de la clase real y el resto)
real_class_probs = y_pred_prob[np.arange(len(y_test)), y_test]
errores = 1.0 - real_class_probs

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
Reporte de Clasificación:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      0.90      0.95        10
   virginica       0.91      1.00      0.95        10

    accuracy                           0.97        30
   macro avg       0.97      0.97      0.97        30
weighted avg       0.97      0.97      0.97        30



```markdown
## 4. Funciones Auxiliares para el Dashboard de Gradio
Generamos las gráficas de manera dinámica para poder renderizarlas en pestañas independientes dentro de Gradio.
```

In [ ]:
def plot_univariate():
    fig, axes = plt.subplots(2, 2, figsize=(10, 8))
    for i, col in enumerate(iris_data.feature_names):
        ax = axes[i // 2, i % 2]
        sns.histplot(data=df, x=col, hue='species_name', kde=True, ax=ax, palette='Set2')
        ax.set_title(f'Distribución de {col}')
    plt.tight_layout()
    return fig

def plot_bivariate():
    fig = plt.figure(figsize=(10, 6))
    sns.scatterplot(data=df, x='sepal length (cm)', y='sepal width (cm)', hue='species_name', palette='Set1', s=70)
    plt.title('Comparación Bivariada: Sepal Length vs Sepal Width')
    return fig

def plot_multivariate():
    # Pairplot no admite retornar una figura directa fácilmente de la misma manera sin llamar a plt.gcf()
    plt.figure()
    g = sns.pairplot(df.drop(columns=['species']), hue='species_name', palette='husl')
    return g.fig

def plot_correlation():
    fig = plt.figure(figsize=(8, 6))
    corr = df[iris_data.feature_names].corr()
    sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
    plt.title('Matriz de Correlación')
    return fig

def plot_metrics():
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Matriz de Confusión
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=iris_data.target_names, yticklabels=iris_data.target_names, ax=axes[0])
    axes[0].set_title('Matriz de Confusión')
    axes[0].set_ylabel('Clase Real')
    axes[0].set_xlabel('Predicción')

    # Histograma de Errores
    sns.histplot(errores, bins=10, kde=True, ax=axes[1], color='crimson')
    axes[1].set_title('Histograma de Errores (1 - Probabilidad Clase Correcta)')
    axes[1].set_xlabel('Magnitud del Error')

    plt.tight_layout()
    return fig

def classify_iris(sl, sw, pl, pw):
    # Clasificar una nueva muestra
    input_data = np.array([[sl, sw, pl, pw]])
    scaled_input = scaler.transform(input_data)
    pred_prob = model.predict(scaled_input)[0]
    class_idx = np.argmax(pred_prob)
    species = iris_data.target_names[class_idx]
    confidence = pred_prob[class_idx] * 100
    return f"Especie Predicha: {species.upper()} (Confianza: {confidence:.2f}%)"

```markdown
## 5. Diseño y Lanzamiento del Dashboard de Gradio
Creamos una estructura limpia con pestañas para el EDA, Modelado, Clasificador Interactivo, Explicación Teórica y Conclusiones.
```

In [ ]:
# Creación de la estructura del dashboard de Gradio
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🌸 Dashboard Analítico e Interactivo: Dataset Iris")

    with gr.Tabs():
        # Pestaña 1: Resumen del Dataframe
        with gr.TabItem("📄 Vista General (DataFrame)"):
            gr.Markdown("### Muestra de las primeras filas del dataset Iris")
            gr.Dataframe(df.head(10))
            gr.Markdown("### Estadísticas Descriptivas")
            gr.Dataframe(df.describe())

        # Pestaña 2: Gráficas de EDA
        with gr.TabItem("📊 Gráficas del EDA"):
            with gr.Tabs():
                with gr.TabItem("Univariadas"):
                    gr.Plot(plot_univariate())
                with gr.TabItem("Bivariadas"):
                    gr.Plot(plot_bivariate())
                with gr.TabItem("Multivariadas"):
                    gr.Plot(plot_multivariate())
                with gr.TabItem("Correlación"):
                    gr.Plot(plot_correlation())

        # Pestaña 3: Modelo y Evaluación
        with gr.TabItem("🧠 Red Neuronal & Evaluación"):
            gr.Markdown("### Reporte de Clasificación")
            gr.Text(value=report)
            gr.Markdown("### Gráficos de Rendimiento (Matriz de Confusión & Histograma de Errores)")
            gr.Plot(plot_metrics())

        # Pestaña 4: Clasificador Interactivo (Muestra de ejemplo)
        with gr.TabItem("🔮 Clasificar Nueva Muestra"):
            gr.Markdown("### Inserta las medidas para clasificar una flor de Iris en tiempo real")
            with gr.Row():
                sl_input = gr.Slider(minimum=4.0, maximum=8.0, value=5.8, step=0.1, label="Sepal Length (cm)")
                sw_input = gr.Slider(minimum=2.0, maximum=4.5, value=3.0, step=0.1, label="Sepal Width (cm)")
                pl_input = gr.Slider(minimum=1.0, maximum=7.0, value=4.35, step=0.1, label="Petal Length (cm)")
                pw_input = gr.Slider(minimum=0.1, maximum=2.5, value=1.3, step=0.1, label="Petal Width (cm)")

            predict_btn = gr.Button("Clasificar")
            output_text = gr.Textbox(label="Resultado de la Red Neuronal")
            predict_btn.click(classify_iris, inputs=[sl_input, sw_input, pl_input, pw_input], outputs=output_text)

        # Pestaña 5: Explicación del Ejercicio
        with gr.TabItem("📝 Explicación del Proyecto"):
            gr.Markdown("""
            ### Descripción de Columnas:
            * **sepal length (cm)**: Longitud del sépalo en centímetros.
            * **sepal width (cm)**: Ancho del sépalo en centímetros.
            * **petal length (cm)**: Longitud del pétalo en centímetros.
            * **petal width (cm)**: Ancho del pétalo en centímetros.
            * **species_name**: Nombre de la clase flor (Setosa, Versicolor, Virginica).

            ### Arquitectura de la Red Neuronal:
            1. **Capa de Entrada**: Recibe las 4 características normalizadas de la flor.
            2. **Capas Ocultas**: Dos capas densas con activación *ReLU* para aprender características complejas y no lineales.
            3. **Capa de Salida**: 3 neuronas con activación *Softmax* para calcular la probabilidad predictiva de cada clase.
            """)

        # Pestaña 6: Conclusiones
        with gr.TabItem("💡 Conclusiones"):
            gr.Markdown("""
            ### Insights y Conclusiones del EDA y Modelado:

            1. **Separabilidad de Clases**: La especie *Setosa* se diferencia de forma perfecta y lineal del resto empleando únicamente el tamaño del pétalo (se observa claramente en las distribuciones univariadas y bivariadas).
            2. **Frontera de Decisión Compleja**: Las especies *Versicolor* y *Virginica* presentan un ligero solapamiento en sus dimensiones de pétalos y sépalos, lo que justifica la utilización de modelos más avanzados como Redes Neuronales.
            3. **Rendimiento del Modelo**: La red neuronal alcanza una precisión (accuracy) excelente, cometiendo muy pocos fallos. El histograma de errores confirma que la mayoría de predicciones se realizan con un nivel de confianza muy alto (error cercano a 0).
            4. **Aplicabilidad Práctica**: Gracias al escalado estandarizado previo, la red neuronal es robusta ante diferentes rangos de entrada ingresados en tiempo real por el usuario en la interfaz interactiva.
            """)

# Lanzar el dashboard interactivo
demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://3a8a59813e6cfd6990.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://3a8a59813e6cfd6990.gradio.live


<Figure size 640x480 with 0 Axes>